# Day 051 — Exercise 2: Widget Input Handling

**What you'll build:** `validate_user_input(text, max_chars)`, `clamp(value, lo, hi)`, and `build_settings(model, temperature, system_prompt)` — the functions that turn raw widget values into safe, validated inputs.

**Why it matters:** Widgets return whatever the user typed or dragged. `st.chat_input` can return empty strings; a `system_prompt` text area can be blank; a temperature restored from session state can be out of range. Validate at the boundary — *before* the value reaches the model — so the UI stays a thin shell over trustworthy data.

## Provided: Setup + Session State (from Exercise 1)

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import ollama


def init_session(state: dict) -> dict:
    """
    Idempotently initialise a Streamlit-style session_state dict.

    Streamlit reruns the WHOLE script top-to-bottom on every interaction, so
    initialisation must never overwrite existing data. Only set a key if absent.

    Ensures keys:
        'messages'  -> list of {'role', 'content'} dicts (starts empty)
        'settings'  -> {'model', 'temperature', 'system_prompt'}
    Returns the same dict, mutated in place.
    """
    if 'messages' not in state:
        state['messages'] = []
    if 'settings' not in state:
        state['settings'] = {
            'model': 'llama3.2',
            'temperature': 0.7,
            'system_prompt': 'You are a helpful assistant.',
        }
    return state


def add_message(state: dict, role: str, content: str) -> dict:
    """Append a {'role', 'content'} message to state['messages']; return it."""
    if role not in ('user', 'assistant', 'system'):
        raise ValueError(f'invalid role: {role!r}')
    msg = {'role': role, 'content': content}
    state['messages'].append(msg)
    return msg


def reset_messages(state: dict) -> None:
    """Clear the conversation but keep settings (a 'Clear chat' button)."""
    state['messages'] = []

## Your Implementation

In [ ]:
def validate_user_input(text: str, max_chars: int = 2000) -> tuple[bool, str]:
    """
    Returns (is_valid, result):
      - empty/whitespace : (False, 'Please enter a message.')
      - too long         : (False, 'Message too long (max N chars).')
      - valid            : (True, cleaned_text)   # stripped
    """
    # TODO: cleaned = text.strip()
    # TODO: if not cleaned: return (False, 'Please enter a message.')
    # TODO: if len(cleaned) > max_chars: return (False, f'Message too long (max {max_chars} chars).')
    # TODO: return (True, cleaned)
    pass


def clamp(value: float, lo: float, hi: float) -> float:
    """Clamp value into [lo, hi]."""
    # TODO: return max(lo, min(hi, value))
    pass


def build_settings(model: str, temperature: float, system_prompt: str) -> dict:
    """temperature clamped to [0,1]; empty system_prompt -> default."""
    # TODO: sp = system_prompt.strip() or 'You are a helpful assistant.'
    # TODO: return {'model': model, 'temperature': float(clamp(temperature, 0.0, 1.0)), 'system_prompt': sp}
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: valid input returns (True, stripped_text)
    try:
        ok, result = validate_user_input('  hello world  ')
        assert ok is True, f'expected valid, got {ok}'
        assert result == 'hello world', f'expected stripped text, got {result!r}'
        passed += 1; print('✅ Check 1: valid input -> (True, stripped)')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: empty / whitespace input is rejected
    try:
        ok1, _ = validate_user_input('')
        ok2, _ = validate_user_input('   ')
        assert ok1 is False and ok2 is False, 'empty/whitespace must be invalid'
        passed += 1; print('✅ Check 2: empty/whitespace rejected')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: over-length input is rejected
    try:
        ok, msg = validate_user_input('x' * 50, max_chars=10)
        assert ok is False, 'over-length input must be invalid'
        assert 'too long' in msg.lower(), f'expected a length message, got {msg!r}'
        passed += 1; print('✅ Check 3: over-length input rejected')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: clamp bounds values correctly
    try:
        assert clamp(-0.5, 0.0, 1.0) == 0.0, 'below-range should clamp to lo'
        assert clamp(1.5, 0.0, 1.0) == 1.0, 'above-range should clamp to hi'
        assert clamp(0.3, 0.0, 1.0) == 0.3, 'in-range should pass through'
        passed += 1; print('✅ Check 4: clamp bounds values into [lo, hi]')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: build_settings clamps temperature + defaults empty prompt
    try:
        s = build_settings('llama3.2', 2.0, '   ')
        assert s['temperature'] == 1.0, f"temperature not clamped: {s['temperature']}"
        assert s['system_prompt'] == 'You are a helpful assistant.', 'empty prompt not defaulted'
        assert s['model'] == 'llama3.2', 'model not carried through'
        passed += 1; print('✅ Check 5: build_settings clamps + defaults correctly')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def validate_user_input(text: str, max_chars: int = 2000) -> tuple[bool, str]:
    """
    Validate raw text from an st.chat_input / st.text_area widget before it is
    sent to the model.

    Returns (is_valid, result):
      - empty/whitespace : (False, 'Please enter a message.')
      - too long         : (False, 'Message too long (max N chars).')
      - valid            : (True, cleaned_text)   # stripped
    """
    cleaned = text.strip()
    if not cleaned:
        return (False, 'Please enter a message.')
    if len(cleaned) > max_chars:
        return (False, f'Message too long (max {max_chars} chars).')
    return (True, cleaned)


def clamp(value: float, lo: float, hi: float) -> float:
    """Clamp a widget value into [lo, hi]. st.slider bounds live input, but a
    value restored from session_state or a URL param may be out of range."""
    return max(lo, min(hi, value))


def build_settings(model: str, temperature: float, system_prompt: str) -> dict:
    """
    Assemble a validated settings dict from sidebar widget values.
    - temperature clamped to [0.0, 1.0]
    - system_prompt stripped; empty falls back to a default
    """
    sp = system_prompt.strip() or 'You are a helpful assistant.'
    return {
        'model': model,
        'temperature': float(clamp(temperature, 0.0, 1.0)),
        'system_prompt': sp,
    }
```

**Why this works:** Returning a `(bool, str)` tuple lets the UI branch cleanly: on `False` show `st.warning(result)`, on `True` send `result` to the model. `clamp` is a one-liner but guards against out-of-range values that slip past the slider (e.g. restored from state). `build_settings` centralises every defaulting rule so the sidebar code stays declarative.
</details>